## 1. import

In [8]:
import numpy as np
import pandas as pd

## 2. config

In [9]:
class CFG:
    SEED = 42
    N_SPLITS = 5
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"

## 3. models dict

In [10]:
# ==============================
# Load OOF / PRED
# ==============================
model_dict = {
    "018": (
        np.load(CFG.NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
    ),
    "024": (
        np.load(CFG.NPY_PATH + "oof_xgb_digit_orderedte_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_xgb_digit_orderedte_min_proba_biased.npy"),
    ),
    "025": (
        np.load(CFG.NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_lgb_digit_te_min_proba_biased.npy"),
    ),
    "026": (
        np.load(CFG.NPY_PATH + "oof_realmlp_pair_te_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_realmlp_pair_te_min_proba_biased.npy"),
    ),
    "033": (
        np.load(CFG.NPY_PATH + "oof_cat_formula6_orig_pair_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_cat_formula6_orig_pair_proba_biased.npy"),
    ),
    "034": (
        np.load(CFG.NPY_PATH + "oof_dao_xgb_pair_te_core_min_proba.npy"),
        np.load(CFG.NPY_PATH + "pred_dao_xgb_pair_te_core_min_proba.npy"),
    ),
    "035": (
        np.load(CFG.NPY_PATH + "oof_dao_xgb_pair_te_plus_formula_min_proba.npy"),
        np.load(CFG.NPY_PATH + "pred_dao_xgb_pair_te_plus_formula_min_proba.npy"),
    ),
    "037": (
        np.load(CFG.NPY_PATH + "oof_dao_xgb_pair_te_posthoc_only_proba.npy"),
        np.load(CFG.NPY_PATH + "pred_dao_xgb_pair_te_posthoc_only_proba.npy"),
    ),
}

## class order fix

In [11]:
perm = [2, 0, 1]

for name in ["018", "024", "025", "026", "034", "035", "037"]:
    oof_t, pred_t = model_dict[name]
    model_dict[name] = (oof_t[:, perm], pred_t[:, perm])

for name, (oof_arr, pred_arr) in model_dict.items():
    print(name, oof_arr.shape, pred_arr.shape, np.round(oof_arr[:3].sum(axis=1), 6))

018 (630000, 3) (270000, 3) [1. 1. 1.]
024 (630000, 3) (270000, 3) [1. 1. 1.]
025 (630000, 3) (270000, 3) [1. 1. 1.]
026 (630000, 3) (270000, 3) [1. 1. 1.]
033 (630000, 3) (270000, 3) [1. 1. 1.]
034 (630000, 3) (270000, 3) [1. 1. 1.]
035 (630000, 3) (270000, 3) [1. 1. 1.]
037 (630000, 3) (270000, 3) [0.579839 0.579903 0.579442]


## 4. show correlations

In [12]:
# ==============================
# Helpers
# ==============================
def flatten_multiclass_proba(arr: np.ndarray) -> np.ndarray:
    """
    (n_samples, n_classes) -> 1d flatten
    相関確認用。まずは全クラスまとめて見る。
    """
    if arr.ndim == 1:
        return arr.astype(np.float64)
    return arr.reshape(-1).astype(np.float64)

def rank01_pd(x: np.ndarray) -> np.ndarray:
    s = pd.Series(x)
    r = s.rank(method="average").values.astype(np.float64)
    denom = max(len(r) - 1, 1)
    return (r - 1.0) / denom

# ==============================
# Build DataFrames
# ==============================
names = list(model_dict.keys())

oof_flat = [flatten_multiclass_proba(model_dict[k][0]) for k in names]
pred_flat = [flatten_multiclass_proba(model_dict[k][1]) for k in names]

oof_lens = [len(x) for x in oof_flat]
pred_lens = [len(x) for x in pred_flat]

if len(set(oof_lens)) != 1:
    raise ValueError(f"OOF lengths mismatch: {dict(zip(names, oof_lens))}")
if len(set(pred_lens)) != 1:
    raise ValueError(f"PRED lengths mismatch: {dict(zip(names, pred_lens))}")

df_oof = pd.DataFrame(np.stack(oof_flat, axis=1), columns=names)
df_pred = pd.DataFrame(np.stack(pred_flat, axis=1), columns=names)

print("df_oof shape :", df_oof.shape)
print("df_pred shape:", df_pred.shape)

# ==============================
# Raw correlation
# ==============================
print("\n=== OOF Pearson corr (raw) ===")
display(df_oof.corr())

print("\n=== TEST Pearson corr (raw) ===")
display(df_pred.corr())

# ==============================
# Rank correlation
# ==============================
df_oof_rank = df_oof.apply(lambda col: rank01_pd(col.values), axis=0, result_type="expand")
df_oof_rank.columns = names

df_pred_rank = df_pred.apply(lambda col: rank01_pd(col.values), axis=0, result_type="expand")
df_pred_rank.columns = names

print("\n=== OOF corr (rank -> Pearson) ===")
display(df_oof_rank.corr())

print("\n=== TEST corr (rank -> Pearson) ===")
display(df_pred_rank.corr())

# ==============================
# Focused view
# ==============================
target = "025"
base_models = [model for model in model_dict.keys() if model != target]

print(f"\n=== Focused correlations vs {target} ===")
for bm in base_models:
    print(
        f"{bm:30s} | "
        f"OOF raw={df_oof[target].corr(df_oof[bm]):.6f} | "
        f"TEST raw={df_pred[target].corr(df_pred[bm]):.6f} | "
        f"OOF rank={df_oof_rank[target].corr(df_oof_rank[bm]):.6f} | "
        f"TEST rank={df_pred_rank[target].corr(df_pred_rank[bm]):.6f}"
    )

df_oof shape : (1890000, 8)
df_pred shape: (810000, 8)

=== OOF Pearson corr (raw) ===


,018,024,025,026,033,034,035,037
018,1.000000,0.995078,0.994775,0.994960,0.989416,0.996317,0.996174,0.930650
024,0.995078,1.000000,0.998641,0.995699,0.992313,0.996957,0.997369,0.930123
025,0.994775,0.998641,1.000000,0.995700,0.990491,0.996357,0.996576,0.931079
026,0.994960,0.995699,0.995700,1.000000,0.989536,0.995773,0.995699,0.931383
033,0.989416,0.992313,0.990491,0.989536,1.000000,0.992423,0.993232,0.917471
034,0.996317,0.996957,0.996357,0.995773,0.992423,1.000000,0.998701,0.932170
035,0.996174,0.997369,0.996576,0.995699,0.993232,0.998701,1.000000,0.930783
037,0.930650,0.930123,0.931079,0.931383,0.917471,0.932170,0.930783,1.000000



=== TEST Pearson corr (raw) ===


,018,024,025,026,033,034,035,037
018,1.000000,0.996178,0.995935,0.995901,0.990009,0.996821,0.996733,0.930685
024,0.996178,1.000000,0.999317,0.997025,0.992681,0.997961,0.998157,0.930948
025,0.995935,0.999317,1.000000,0.997172,0.991018,0.997460,0.997475,0.931944
026,0.995901,0.997025,0.997172,1.000000,0.990242,0.996689,0.996514,0.931936
033,0.990009,0.992681,0.991018,0.990242,1.000000,0.993034,0.993585,0.917518
034,0.996821,0.997961,0.997460,0.996689,0.993034,1.000000,0.999029,0.931611
035,0.996733,0.998157,0.997475,0.996514,0.993585,0.999029,1.000000,0.930631
037,0.930685,0.930948,0.931944,0.931936,0.917518,0.931611,0.930631,1.000000



=== OOF corr (rank -> Pearson) ===


,018,024,025,026,033,034,035,037
018,1.000000,0.942156,0.945456,0.911784,0.893168,0.938409,0.937743,0.941416
024,0.942156,1.000000,0.992057,0.908159,0.960458,0.985251,0.979616,0.979720
025,0.945456,0.992057,1.000000,0.909005,0.951944,0.978616,0.972389,0.976969
026,0.911784,0.908159,0.909005,1.000000,0.885193,0.903430,0.901127,0.904928
033,0.893168,0.960458,0.951944,0.885193,1.000000,0.961895,0.963662,0.947266
034,0.938409,0.985251,0.978616,0.903430,0.961895,1.000000,0.989002,0.985549
035,0.937743,0.979616,0.972389,0.901127,0.963662,0.989002,1.000000,0.974192
037,0.941416,0.979720,0.976969,0.904928,0.947266,0.985549,0.974192,1.000000



=== TEST corr (rank -> Pearson) ===


,018,024,025,026,033,034,035,037
018,1.000000,0.945409,0.952764,0.952204,0.900769,0.940429,0.939614,0.943556
024,0.945409,1.000000,0.996804,0.928774,0.967857,0.988288,0.982775,0.982569
025,0.952764,0.996804,1.000000,0.937726,0.960519,0.983981,0.977426,0.982837
026,0.952204,0.928774,0.937726,1.000000,0.879834,0.913080,0.905019,0.926132
033,0.900769,0.967857,0.960519,0.879834,1.000000,0.969307,0.971383,0.954116
034,0.940429,0.988288,0.983981,0.913080,0.969307,1.000000,0.990169,0.985740
035,0.939614,0.982775,0.977426,0.905019,0.971383,0.990169,1.000000,0.975309
037,0.943556,0.982569,0.982837,0.926132,0.954116,0.985740,0.975309,1.000000



=== Focused correlations vs 025 ===
018                            | OOF raw=0.994775 | TEST raw=0.995935 | OOF rank=0.945456 | TEST rank=0.952764
024                            | OOF raw=0.998641 | TEST raw=0.999317 | OOF rank=0.992057 | TEST rank=0.996804
026                            | OOF raw=0.995700 | TEST raw=0.997172 | OOF rank=0.909005 | TEST rank=0.937726
033                            | OOF raw=0.990491 | TEST raw=0.991018 | OOF rank=0.951944 | TEST rank=0.960519
034                            | OOF raw=0.996357 | TEST raw=0.997460 | OOF rank=0.978616 | TEST rank=0.983981
035                            | OOF raw=0.996576 | TEST raw=0.997475 | OOF rank=0.972389 | TEST rank=0.977426
037                            | OOF raw=0.931079 | TEST raw=0.931944 | OOF rank=0.976969 | TEST rank=0.982837


In [13]:
# 3x3 の classwise cross-corr を見る
for bm in base_models:
    print(f"\n=== {bm} vs {target} : OOF classwise cross-corr ===")
    a = model_dict[bm][0]   # base model OOF
    b = model_dict[target][0]  # target model OOF

    mat = np.zeros((a.shape[1], b.shape[1]))
    for i in range(a.shape[1]):
        for j in range(b.shape[1]):
            mat[i, j] = np.corrcoef(a[:, i], b[:, j])[0, 1]

    display(pd.DataFrame(
        mat,
        index=[f"{bm}_class{i}" for i in range(a.shape[1])],
        columns=[f"{target}_class{j}" for j in range(b.shape[1])]
    ))


=== 018 vs 025 : OOF classwise cross-corr ===


,025_class0,025_class1,025_class2
018_class0,0.971902,-0.254501,-0.123702
018_class1,-0.287913,0.996963,-0.921963
018_class2,-0.086618,-0.920920,0.992550



=== 024 vs 025 : OOF classwise cross-corr ===


,025_class0,025_class1,025_class2
024_class0,0.990794,-0.272963,-0.112047
024_class1,-0.290260,0.999520,-0.923685
024_class2,-0.087583,-0.925831,0.998044



=== 026 vs 025 : OOF classwise cross-corr ===


...

,025_class0,025_class1,025_class2
034_class0,0.979043,-0.257965,-0.122952
034_class1,-0.289418,0.998162,-0.922610
034_class2,-0.078456,-0.926295,0.994879



=== 035 vs 025 : OOF classwise cross-corr ===


,025_class0,025_class1,025_class2
035_class0,0.978528,-0.257373,-0.123362
035_class1,-0.289192,0.998537,-0.923090
035_class2,-0.078523,-0.926567,0.995189



=== 037 vs 025 : OOF classwise cross-corr ===


,025_class0,025_class1,025_class2
037_class0,0.979043,-0.257965,-0.122952
037_class1,-0.289418,0.998162,-0.922610
037_class2,-0.078456,-0.926295,0.994879


In [14]:
# 行和を確認する
for name, (oof, pred) in model_dict.items():
    print(name)
    print("  oof shape:", oof.shape)
    print("  pred shape:", pred.shape)
    print("  oof row sum head:", np.round(oof[:5].sum(axis=1), 6))
    print("  pred row sum head:", np.round(pred[:5].sum(axis=1), 6))
    print("  oof min/max:", float(oof.min()), float(oof.max()))
    print()

018
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 1.9344165225501327e-16 1.0

024
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 3.1933023818319686e-11 0.9999999093325029

025
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 4.440628106685934e-11 0.9999999066650522

026
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 2.7523189244835606e-05 0.9999186461608099

033
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 3.287685263082626e-07 0.9999699110210166

034
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1